# 도구 호출(Tool Calling) 기초

LLM은 글을 만들 뿐, 파일에 저장하거나 메일을 보내지는 못합니다.
**도구 호출**은 이 한계를 넘는 방법입니다. LLM이 "이 함수를 이런 값으로 불러 주세요"라고 요청하면,
우리 코드가 그 함수를 대신 실행하고 결과를 다시 알려 줍니다.

이 노트북에서는 **상담 도중 사용자의 이름·전화번호·주소를 받아 파일에 저장하는** 챗봇을 만듭니다.

## 전체 흐름

```
① 파이썬 함수를 만든다              record_personal_info_tool()
② LLM에게 그 함수를 설명한다         JSON 스키마
③ LLM이 "이 함수를 부르세요"라고 답한다   finish_reason == "tool_calls"
④ 우리 코드가 함수를 실행한다
⑤ 결과를 LLM에게 돌려주고 최종 답변을 받는다
```

**중요**: LLM이 함수를 직접 실행하는 것이 아닙니다. **실행은 항상 우리 코드가 합니다.**
LLM은 "무엇을, 어떤 값으로 부를지"만 정합니다.

> ⚠️ 실습에는 **가짜 개인정보**를 쓰세요. 실제 이름·전화번호·주소를 입력하면 `personal_info.txt`에 그대로 저장됩니다.

## Step 1. 준비

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import gradio as gr
#from IPython.display import Markdown, display

load_dotenv(override=True)
openai = OpenAI()

## Step 2. 시스템 프롬프트 — 도구를 언제 쓸지 알려 주기

도구를 등록하는 것만으로는 부족합니다. **언제 부를지**를 프롬프트로 알려 줘야 합니다.

아래 프롬프트에는 규칙이 두 개 들어 있습니다.

1. 이름·전화번호·주소 **세 가지를 모두 받으면** 도구를 호출한다.
2. 다 모이기 전에는 **호출하지 말고** 빠진 항목을 물어본다.

두 번째 규칙이 없으면 모델이 빈 값으로 도구를 부르거나, 값을 지어내기도 합니다.

In [34]:
system_prompt = """
당신은 대화 전문가이며 대화하는 사람의 고민을 들어주고 제안을 할 수 있습니다.

대화 중에 상담 기록을 남기기 위해 사용자의 이름, 전화번호, 주소를 물어보세요.
세 가지를 모두 받으면 record_personal_info_tool 도구를 호출해서 기록하세요.
세 가지가 다 모이기 전에는 도구를 호출하지 말고, 빠진 항목을 물어보세요.
"""

## Step 3. 먼저 도구 없이 대화해 보기

비교를 위해 도구를 주지 않고 한 번 호출해 봅니다.
프롬프트에 "기록하라"고 적혀 있어도, **도구가 없으면 모델은 말만 할 뿐 아무것도 저장하지 못합니다.**

In [35]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "나의 고민을 해결할 수 있는 방법을 알려 주세요"},
]

In [36]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
print(response.choices[0].message.content)

물론이죠. 어떤 고민인지 편하게 말씀해 주세요.  
제가 함께 정리하고 해결 방법을 찾아드릴게요.

상담 기록을 위해 **이름, 전화번호, 주소**도 알려주시면 좋습니다.  
지금 바로 보내주셔도 되고, 고민을 먼저 말씀해주셔도 괜찮아요.


## Step 4. 도구로 쓸 파이썬 함수 만들기

특별한 것은 없습니다. **그냥 평범한 파이썬 함수**입니다.

- 파일에 한 줄씩 덧붙여 저장합니다(`"a"` 모드).
- `print`를 넣어 두면 도구가 실제로 불렸는지 눈으로 확인할 수 있습니다.
- 마지막에 **문자열을 돌려줍니다.** 이 값이 나중에 LLM에게 "작업 결과"로 전달됩니다.

다음 셀에서 함수를 직접 호출해 정상 동작을 먼저 확인합니다. LLM과 연결하기 전에
**함수만 따로 시험해 보는 것**이 문제를 줄이는 방법입니다.

In [37]:
def record_personal_info_tool(name, phone, address):
    print(f"Tool called to record: {name} / {phone} / {address}")
    with open("personal_info.txt", "a", encoding="utf-8") as f:
        f.write(f"{name}\t{phone}\t{address}\n")
    return "personal info is recorded"

In [38]:
record_personal_info_tool("홍길동", "(051) 777-9999", "부산시 해운대구")

Tool called to record: 홍길동 / (051) 777-9999 / 부산시 해운대구


'personal info is recorded'

## Step 5. LLM에게 도구를 설명하기 (JSON 스키마)

### 방법 A: 스키마를 직접 만들기

> Step 8에서 같은 도구를 **자동으로** 만드는 방법(방법 B)과 비교합니다. 먼저 손으로 만들어 봅니다.

LLM은 우리 파이썬 코드를 볼 수 없습니다. 그래서 **함수 설명서**를 JSON으로 만들어 넘겨야 합니다.

| 항목 | 역할 |
|---|---|
| `name` | 함수 이름. 실제 파이썬 함수 이름과 같게 맞추면 관리하기 쉽습니다 |
| `description` | **이 도구를 언제 쓰는지.** 모델이 호출 여부를 판단하는 근거입니다 |
| `properties` | 각 인자의 이름·타입·설명 |
| `required` | 반드시 있어야 하는 인자 |

`description`을 성의 없이 적으면 모델이 도구를 엉뚱한 때에 부르거나 아예 부르지 않습니다.
**설명서가 곧 사용 설명**이라고 생각하세요.

In [39]:
record_personal_info_tool_json = {
    "name": "record_personal_info_tool",
    "description": "Use this tool to record the name, phone number and address that a user provided",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string", "description": "The name of this user"},
            "phone": {"type": "string", "description": "The phone number of this user"},
            "address": {"type": "string", "description": "The address of this user"}
        },
        "required": ["name", "phone", "address"],
        "additionalProperties": False
    }
}

도구는 여러 개를 등록할 수 있으므로 **리스트**로 감쌉니다.
각 항목은 `{"type": "function", "function": 설명서}` 형태입니다.

In [40]:
tools = [{"type": "function", "function": record_personal_info_tool_json}]

In [41]:
tools

[{'type': 'function',
  'function': {'name': 'record_personal_info_tool',
   'description': 'Use this tool to record the name, phone number and address that a user provided',
   'parameters': {'type': 'object',
    'properties': {'name': {'type': 'string',
      'description': 'The name of this user'},
     'phone': {'type': 'string',
      'description': 'The phone number of this user'},
     'address': {'type': 'string', 'description': 'The address of this user'}},
    'required': ['name', 'phone', 'address'],
    'additionalProperties': False}}}]

## Step 6. 도구 호출 처리하기 (이 노트북의 핵심)

여기서 LLM을 **두 번** 호출합니다. 도구 호출은 한 번의 대화로 끝나지 않기 때문입니다.

```
1차 호출  → 모델이 "record_personal_info_tool을 이 값으로 불러 주세요"라고 답함
           (finish_reason == "tool_calls")
   ↓
우리 코드가 함수를 실행
   ↓
2차 호출  → 실행 결과를 붙여서 다시 물어봄 → 모델이 사용자에게 할 최종 답변을 만듦
```

코드를 한 줄씩 보면 이렇습니다.

| 코드 | 하는 일 |
|---|---|
| `tools=tools` | 이번 요청에 쓸 수 있는 도구 목록을 함께 보냄 |
| `finish_reason == "tool_calls"` | 모델이 답변 대신 **도구 호출을 요청**했는지 확인 |
| `messages.append(assistant_message)` | 모델의 호출 요청을 대화 기록에 남김 (**빠뜨리면 오류**) |
| `json.loads(tool_call.function.arguments)` | 모델이 보낸 인자는 **JSON 문자열**이라 파싱이 필요 |
| `{"role": "tool", ...}` | 실행 결과를 `tool` 역할로 추가. `tool_call_id`로 어느 호출의 결과인지 연결 |

`tool_calls`가 아니면 보통의 답변이므로 그대로 돌려줍니다.

In [42]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = record_personal_info_tool(
                args.get("name"), args.get("phone"), args.get("address")
            )
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})

        response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)

    return response.choices[0].message.content

## Step 7. 실행해 보기

Gradio 챗봇으로 실제 대화를 해 봅니다. 이렇게 진행해 보세요.

1. "요즘 고민이 있어요" 처럼 아무 말이나 겁니다.
2. 모델이 이름·전화번호·주소를 물어봅니다.
3. **일부만** 알려 줍니다 → 도구를 부르지 않고 나머지를 물어보는지 확인하세요.
4. 나머지를 마저 알려 줍니다 → 노트북에 `Tool called to record: ...`가 찍히면 성공입니다.
5. `personal_info.txt` 파일을 열어 저장된 내용을 확인합니다.

> 다시 말씀드리지만 **가짜 정보**를 쓰세요.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Step 8. 같은 도구를 `@function_tool`로 만들기 — 두 방식 비교

지금까지 한 방식에는 불편한 점이 있습니다. **함수와 JSON 스키마를 따로 관리**해야 합니다.

- 함수에 인자를 하나 추가하면 → 스키마도 고쳐야 합니다.
- 고치는 것을 잊으면 → 오류는 안 나고, 모델이 그 인자를 **영원히 채우지 않습니다.**

OpenAI **Agents SDK**의 `@function_tool` 데코레이터는 이 문제를 없앱니다.
함수의 **타입 힌트와 독스트링을 읽어 스키마를 자동으로 만들어** 줍니다.

> 설치: `pip install openai-agents`

In [ ]:
from agents import Agent, Runner, function_tool

### 방법 B: 데코레이터로 만들기

함수 위에 `@function_tool`만 붙이면 끝입니다. 대신 두 가지를 지켜야 합니다.

1. **타입 힌트**(`name: str`)를 반드시 적습니다 → 스키마의 `type`이 됩니다.
2. **독스트링**에 설명과 `Args:`를 적습니다 → `description`이 됩니다.

In [ ]:
@function_tool
def record_personal_info_tool_v2(name: str, phone: str, address: str) -> str:
    """
    사용자가 알려 준 이름, 전화번호, 주소를 파일에 기록한다.

    Args:
        name: 사용자 이름
        phone: 사용자 전화번호
        address: 사용자 주소
    """
    print(f"Tool called to record: {name} / {phone} / {address}")
    with open("personal_info.txt", "a", encoding="utf-8") as f:
        f.write(f"{name}\t{phone}\t{address}\n")
    return "personal info is recorded"

In [ ]:
# 자동으로 만들어진 스키마를 확인해 봅니다.
print("이름:", record_personal_info_tool_v2.name)
print("설명:", record_personal_info_tool_v2.description)
print()
print(json.dumps(record_personal_info_tool_v2.params_json_schema, indent=2, ensure_ascii=False))

### 무엇이 어디서 왔나

Step 5에서 손으로 쓴 것과 비교해 보세요. **내용이 같습니다.**

| 스키마 항목 | 방법 A (손으로) | 방법 B (데코레이터) |
|---|---|---|
| `name` | 직접 입력 | 함수 이름에서 자동 |
| `description` | 직접 입력 | **독스트링 첫 줄**에서 자동 |
| `type: "string"` | 직접 입력 | **타입 힌트 `: str`** 에서 자동 |
| 인자 설명 | 직접 입력 | 독스트링의 **`Args:`** 에서 자동 |
| `required` | 직접 입력 | 기본값 없는 인자에서 자동 |

실행해 보면 `title` 같은 항목이 몇 개 더 붙어 있는데, 모델이 읽는 데는 문제가 없습니다.

### 진짜 큰 차이는 스키마가 아닙니다

데코레이터의 이점은 "타이핑이 준다"는 정도지만, **Agents SDK를 쓰면 Step 6의 코드가 통째로 사라집니다.**

Step 6에서 우리가 손으로 한 일을 떠올려 보세요.

1. `finish_reason == "tool_calls"` 인지 확인하고
2. `arguments`를 `json.loads`로 풀고
3. 함수를 실행하고
4. 결과를 `{"role": "tool", ...}`로 붙이고
5. LLM을 **다시** 호출

이 다섯 단계를 **SDK가 알아서 합니다.** 우리는 도구를 등록하고 실행만 시키면 됩니다.

In [ ]:
agent = Agent(
    name="상담 도우미",
    instructions=system_prompt,        # Step 2에서 만든 것을 그대로 사용
    model="gpt-5.4-mini",
    tools=[record_personal_info_tool_v2],
)

# 정보가 부족할 때 — 도구를 부르지 않고 물어봐야 한다
result = await Runner.run(agent, "요즘 고민이 많아요. 제 이름은 김철수입니다.")
print(result.final_output)

In [ ]:
# 세 가지를 모두 줄 때 — 도구를 부른다
result = await Runner.run(agent, "이름 김철수, 전화 010-1234-5678, 주소 서울시 강남구입니다.")
print(result.final_output)

### Gradio에 붙이기

Step 6의 `chat()` 함수와 비교해 보세요. **한 줄로 줄었습니다.**
`Runner.run`은 비동기 함수이므로 `async def`로 정의하고 `await`를 씁니다.

In [ ]:
def to_agent_input(history, message):
    """gradio 기록을 에이전트 입력으로 바꾼다. role/content만 남긴다.

    gradio는 options, metadata 같은 표시용 키를 함께 넘기는데,
    그대로 보내면 OpenAI API가 거부한다(Unknown parameter: 'input[0].options').
    """
    items = []
    for msg in history:
        role = msg.get("role")
        content = msg.get("content")
        if role in ("user", "assistant") and isinstance(content, str):
            items.append({"role": role, "content": content})
    items.append({"role": "user", "content": message})
    return items


async def chat_with_agent(message, history):
    result = await Runner.run(agent, to_agent_input(history, message))
    return result.final_output


gr.ChatInterface(chat_with_agent).launch(inbrowser=True)

### 두 방식 비교 정리

| | 방법 A: 직접 만들기 | 방법 B: Agents SDK |
|---|---|---|
| 스키마 | 손으로 JSON 작성 | 타입 힌트·독스트링에서 자동 생성 |
| 함수와 스키마 동기화 | **직접 챙겨야 함** | 자동 |
| 도구 실행 루프 | 직접 구현 (약 15줄) | SDK가 처리 |
| 여러 번 도구 호출 | 직접 반복문 작성 | SDK가 처리 |
| 추가 설치 | 없음 (`openai`만) | `openai-agents` 필요 |
| 다른 LLM으로 교체 | 같은 방식으로 가능 | SDK 지원 범위 안에서 |
| 내부 동작 이해 | **잘 보임** | 감춰짐 |

**어느 쪽을 쓰면 되나**

- **배울 때는 방법 A.** 도구 호출이 실제로 어떻게 오가는지 봐야 문제가 생겼을 때 고칠 수 있습니다.
- **만들 때는 방법 B.** 도구가 늘어날수록 손으로 쓴 스키마는 관리하기 어렵습니다.

방법 A를 먼저 해 본 지금은, 방법 B가 **무엇을 대신 해 주고 있는지** 알고 쓰는 셈입니다.
`openai_agent_code.ipynb`에서 이어서 다룹니다.

## 정리

| 개념 | 설명 |
|---|---|
| 도구 호출 | LLM이 "이 함수를 불러 달라"고 **요청**하는 것. 실행은 우리 코드가 한다 |
| JSON 스키마 | LLM에게 주는 함수 설명서. `description`이 호출 판단의 근거 |
| `finish_reason` | `"tool_calls"`면 도구 호출 요청, 아니면 보통 답변 |
| 2단계 호출 | 도구 실행 결과를 붙여 **다시 물어봐야** 최종 답변이 나온다 |
| `@function_tool` | 타입 힌트·독스트링에서 스키마를 자동 생성. 실행 루프도 SDK가 처리 |

**기억할 것**

- LLM은 함수를 실행하지 못합니다. **무엇을 부를지만** 정합니다.
- 언제 부를지는 **시스템 프롬프트**로, 어떻게 부를지는 **JSON 스키마**로 알려 줍니다.
- 도구가 실제로 불렸는지 확인하려면 함수 안에 `print`를 넣어 두세요.

## 스스로 해 보기

1. `description`을 `"도구"`처럼 성의 없이 바꾸고 다시 대화해 보세요. 모델이 제대로 호출하나요?
2. 시스템 프롬프트에서 "다 모이기 전에는 호출하지 말라"는 문장을 지워 보세요. 무엇이 달라지나요?
3. 상담 내용을 요약해 저장하는 두 번째 도구를 만들어 `tools` 목록에 추가해 보세요.
4. 지금 코드는 도구 호출을 **한 번만** 처리합니다. 2차 응답에서 모델이 또 도구를 부르면 어떻게 될까요?
   `while` 문으로 바꿔 여러 번 처리하도록 고쳐 보세요.